# GraphRAG vs Plain RAG — Head-to-Head Comparison

Run this notebook after both `GraphRAG.ipynb` and `Plain_RAG/Plain_RAG.ipynb`
have completed their benchmarks and saved results to the `results/` directory.

**What is held constant (controlled variables):**
- Embedding model: `all-MiniLM-L6-v2`
- LLM: `deepseek-r1:8b` via Ollama
- Final retrieved documents fed to LLM: top-3
- System prompt (word-for-word identical)
- Answer extraction (`FuzzyEvaluator`)
- Evaluation dataset: `pqa_labeled` (same N samples, same order)

**What differs (the independent variable):**
- **GraphRAG**: ArangoDB + wide search (top-75) + CrossEncoder reranking + graph expansion
- **Plain RAG**: FAISS + direct top-3 retrieval + concatenated chunk text

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)

RESULTS_DIR   = 'results'
GRAPHRAG_FILE = os.path.join(RESULTS_DIR, 'graphrag_results.json')
PLAINRAG_FILE = os.path.join(RESULTS_DIR, 'plainrag_results.json')
LABELS        = ['yes', 'no', 'maybe']

for path in [GRAPHRAG_FILE, PLAINRAG_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{path} not found. Run the corresponding notebook first.'
        )

with open(GRAPHRAG_FILE) as f:
    gr = json.load(f)
with open(PLAINRAG_FILE) as f:
    pr = json.load(f)

print(f"GraphRAG  : {gr['samples']} samples,  accuracy={gr['accuracy']:.2%},  avg_latency={gr['avg_latency']:.1f}s")
print(f"Plain RAG : {pr['samples']} samples,  accuracy={pr['accuracy']:.2%},  avg_latency={pr['avg_latency']:.1f}s")

## Summary metrics

In [ ]:
models = [gr['model'], pr['model']]
accs   = [gr['accuracy'] * 100, pr['accuracy'] * 100]
lats   = [gr['avg_latency'], pr['avg_latency']]

gr_f1 = f1_score(gr['y_true'], gr['y_pred'], labels=LABELS, average='macro', zero_division=0)
pr_f1 = f1_score(pr['y_true'], pr['y_pred'], labels=LABELS, average='macro', zero_division=0)
f1s   = [gr_f1 * 100, pr_f1 * 100]

summary = pd.DataFrame({
    'Model':           models,
    'Accuracy (%)':    [round(a, 2) for a in accs],
    'Macro F1 (%)':    [round(f, 2) for f in f1s],
    'Avg Latency (s)': [round(l, 2) for l in lats],
    'Samples':         [gr['samples'], pr['samples']],
})
summary = summary.set_index('Model')
print(summary.to_string())

## Accuracy & Latency comparison

In [ ]:
colours = ['#2196F3', '#FF9800']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy
bars = axes[0].bar(models, accs, color=colours, width=0.4, edgecolor='white')
axes[0].set_ylim(0, 100)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy')
for bar, val in zip(bars, accs):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, val + 1.5,
        f'{val:.1f}%', ha='center', fontweight='bold'
    )

# Macro F1
bars2 = axes[1].bar(models, f1s, color=colours, width=0.4, edgecolor='white')
axes[1].set_ylim(0, 100)
axes[1].set_ylabel('Macro F1 (%)')
axes[1].set_title('Macro F1 Score')
for bar, val in zip(bars2, f1s):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, val + 1.5,
        f'{val:.1f}%', ha='center', fontweight='bold'
    )

# Latency
bars3 = axes[2].bar(models, lats, color=colours, width=0.4, edgecolor='white')
axes[2].set_ylabel('Avg latency (s / query)')
axes[2].set_title('Latency')
for bar, val in zip(bars3, lats):
    axes[2].text(
        bar.get_x() + bar.get_width() / 2, val * 1.03,
        f'{val:.1f}s', ha='center', fontweight='bold'
    )

plt.suptitle(
    f'GraphRAG vs Plain RAG  (n={gr["samples"]} samples each)',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

## Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, res, colour in zip(axes, [gr, pr], ['Blues', 'Oranges']):
    cm = confusion_matrix(res['y_true'], res['y_pred'], labels=LABELS)
    sns.heatmap(cm, annot=True, fmt='d', cmap=colour,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(res['model'] + f"  (acc={res['accuracy']:.2%})")

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Per-class F1 comparison

In [ ]:
def per_class_f1(res):
    f1s = f1_score(res['y_true'], res['y_pred'],
                   labels=LABELS, average=None, zero_division=0)
    return dict(zip(LABELS, f1s))

gr_f1s = per_class_f1(gr)
pr_f1s = per_class_f1(pr)

x     = np.arange(len(LABELS))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, [gr_f1s[l] * 100 for l in LABELS],
               width, label=gr['model'], color='#2196F3', edgecolor='white')
bars2 = ax.bar(x + width / 2, [pr_f1s[l] * 100 for l in LABELS],
               width, label=pr['model'], color='#FF9800', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_ylabel('F1 Score (%)')
ax.set_ylim(0, 100)
ax.set_title('Per-class F1 Score')
ax.legend()

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1, f'{h:.1f}', ha='center', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1, f'{h:.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Prediction distribution

In [ ]:
def count_preds(res):
    s = pd.Series(res['y_pred'])
    return s.value_counts().reindex(LABELS, fill_value=0)

gr_counts = count_preds(gr)
pr_counts = count_preds(pr)
gt_counts = pd.Series(gr['y_true']).value_counts().reindex(LABELS, fill_value=0)

x     = np.arange(len(LABELS))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, gt_counts,    width, label='Ground Truth', color='#4CAF50', edgecolor='white')
ax.bar(x,         gr_counts,    width, label=gr['model'],    color='#2196F3', edgecolor='white')
ax.bar(x + width, pr_counts,    width, label=pr['model'],    color='#FF9800', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_ylabel('Count')
ax.set_title('Prediction Distribution vs Ground Truth')
ax.legend()
plt.tight_layout()
plt.show()

## Verdict

In [ ]:
acc_delta = (gr['accuracy'] - pr['accuracy']) * 100
f1_delta  = (gr_f1 - pr_f1) * 100
lat_delta = gr['avg_latency'] - pr['avg_latency']

winner = gr['model'] if acc_delta >= 0 else pr['model']

print('=' * 55)
print(f'  Winner by accuracy : {winner}')
print(f'  Accuracy delta     : {acc_delta:+.2f} pp  (GraphRAG - Plain RAG)')
print(f'  Macro F1 delta     : {f1_delta:+.2f} pp')
print(f'  Latency delta      : {lat_delta:+.2f}s   (GraphRAG - Plain RAG)')
print('=' * 55)
print()
if acc_delta > 0:
    print(
        f'GraphRAG outperforms Plain RAG by {acc_delta:.1f} percentage points, '
        'confirming that the graph-structured context (full abstract reconstruction '
        'via AQL traversal) and CrossEncoder reranking provide measurable improvements '
        'over flat vector retrieval — even with the same embedding model and LLM.'
    )
elif acc_delta < 0:
    print(
        f'Plain RAG outperforms GraphRAG by {abs(acc_delta):.1f} percentage points '
        'on this sample. This may indicate that graph expansion introduces noise for '
        'some query types, or that the sample size is insufficient for a conclusive result.'
    )
else:
    print('Both models perform identically on this sample.')